In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/text-autocomplete/data.zip ./

!unzip -o data.zip

!ls -la
!ls -la data/
!ls -la src/

In [ ]:
!pip install rouge_score

In [ ]:
import sys
sys.path.append('.')

from src.data_utils import load_data, prepare_dataset, split_data
from src.dataset import NextTokenDataset
from src.lstm_model import LSTMAutocomplete
from src.train_lstm import train_lstm
from src.evaluate import evaluate_transformer, evaluate_lstm_model

import torch
import matplotlib.pyplot as plt

print("Все модули загружены!")

In [ ]:
!unzip -o /content/drive/MyDrive/data.zip


train_dataset = NextTokenDataset('data/train.csv', max_len=20)
val_dataset = NextTokenDataset('data/val.csv', max_len=20)
test_dataset = NextTokenDataset('data/test.csv', max_len=20)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=256, shuffle=False, num_workers=2
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, shuffle=True, num_workers=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")

model = LSTMAutocomplete(
    vocab_size=train_dataset.vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    pad_idx=train_dataset.pad_idx
).to(device)

print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_losses, val_losses, rouge_scores = train_lstm(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    dataset=train_dataset,
    device=device,
    epochs=10,
    lr=0.001
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
rouge1 = [r['rouge1'] for r in rouge_scores]
rouge2 = [r['rouge2'] for r in rouge_scores]
plt.plot(rouge1, label='ROUGE-1', marker='o')
plt.plot(rouge2, label='ROUGE-2', marker='o')
plt.xlabel('Epoch')
plt.ylabel('ROUGE Score')
plt.title('ROUGE Scores')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
improvement = [rouge_scores[i]['rouge1'] - rouge_scores[0]['rouge1'] for i in range(len(rouge_scores))]
plt.bar(range(1, len(improvement) + 1), improvement)
plt.xlabel('Epoch')
plt.ylabel('ROUGE-1 Improvement')
plt.title('Improvement over Baseline')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Лучший ROUGE-1: {max([r['rouge1'] for r in rouge_scores]):.4f} на эпохе {rouge_scores.index(max(rouge_scores, key=lambda x: x['rouge1'])) + 1}")

In [ ]:
def test_model_completion(model, dataset, prefix_text, max_tokens=10):

    prefix_tokens = prefix_text.lower().split()
    prefix_ids = dataset.text_to_ids(prefix_tokens)

    generated_ids = model.generate(prefix_ids, max_new_tokens=max_tokens)
    full_text = dataset.ids_to_text(generated_ids)

    return full_text

# Тестовые данные
test_prefixes = [
    "i am going to",
    "today is a",
    "the weather is",
    "i really like",
    "can you help me",
    "what do you think about"
]

print("="*60)
print("ТЕСТИРОВАНИЕ МОДЕЛИ НА НОВЫХ ПРИМЕРАХ")
print("="*60)

for prefix in test_prefixes:
    completion = test_model_completion(model, train_dataset, prefix)
    print(f"\n Префикс: {prefix}")
    print(f"   Продолжение: {completion}")

In [ ]:

print("Запускаем DistilGPT2...")
rouge_tf, tf_preds, tf_refs = evaluate_transformer(
    val_dataset,
    num_samples=200
)

print(f"ROUGE-1: {rouge_tf['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_tf['rouge2']:.4f}")

In [ ]:
from src.evaluate import evaluate_lstm_model

print("="*60)
print("ЗАПУСК ОЦЕНКИ LSTM МОДЕЛИ")
print("="*60)

lstm_rouge, lstm_preds, lstm_refs = evaluate_lstm_model(
    model, val_loader, train_dataset, device, num_examples=50
)

print(f"\n Результаты LSTM:")
print(f"   ROUGE-1: {lstm_rouge['rouge1']:.4f}")
print(f"   ROUGE-2: {lstm_rouge['rouge2']:.4f}")

In [ ]:
from src.evaluate import compare_models

print("="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)

compare_models(
    lstm_rouge=lstm_rouge,
    transformer_rouge=rouge_tf,
    lstm_examples=(lstm_preds, lstm_refs),
    transformer_examples=(tf_preds, tf_refs)
)

In [ ]:
print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ НА TEST ДАТАСЕТЕ")
print("="*60)
test_rouge, test_preds, test_refs = evaluate_lstm_model(
    model, test_loader, train_dataset, device, num_examples=50
)
print(f"LSTM на тесте - ROUGE-1: {test_rouge['rouge1']:.4f}")

In [ ]:
import torch
import pickle
import os
import numpy as np

os.makedirs('models', exist_ok=True)

print("="*60)
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ LSTM И TRANSFORMER")
print("="*60)

print("\n LSTM МОДЕЛЬ:")

best_epoch_idx = np.argmax([r['rouge1'] for r in rouge_scores])
best_epoch_num = best_epoch_idx + 1
best_rouge1 = rouge_scores[best_epoch_idx]['rouge1']
best_rouge2 = rouge_scores[best_epoch_idx]['rouge2']

print(f"   Лучшая эпоха: {best_epoch_num}")
print(f"   ROUGE-1: {best_rouge1:.4f}")
print(f"   ROUGE-2: {best_rouge2:.4f}")

torch.save(model.state_dict(), 'models/lstm_model_best.pth')
print("    LSTM модель сохранена: lstm_model_best.pth")

print("\n TRANSFORMER МОДЕЛЬ (DistilGPT2):")

try:
    transformer_exists = 'rouge_tf' in dir() or 'transformer_rouge' in dir()
    
    if transformer_exists:
        if 'transformer_rouge' in dir():
            tf_rouge1 = transformer_rouge['rouge1']
            tf_rouge2 = transformer_rouge['rouge2']
            tf_preds = transformer_preds if 'transformer_preds' in dir() else []
            tf_refs = transformer_refs if 'transformer_refs' in dir() else []
        else:
            tf_rouge1 = rouge_tf['rouge1']
            tf_rouge2 = rouge_tf['rouge2']
            tf_preds = tf_preds if 'tf_preds' in dir() else []
            tf_refs = tf_refs if 'tf_refs' in dir() else []
        
        print(f"   ROUGE-1: {tf_rouge1:.4f}")
        print(f"   ROUGE-2: {tf_rouge2:.4f}")
        print(f"    Результаты трансформера получены")
    else:
        raise NameError
except:
    print("    Результаты трансформера не найдены")
    tf_rouge1 = 0.0
    tf_rouge2 = 0.0
    tf_preds = []
    tf_refs = []

with open('models/vocab.pkl', 'wb') as f:
    pickle.dump(train_dataset.vocab, f)
print("\n Словарь сохранен: vocab.pkl")

lstm_results = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'rouge_scores': rouge_scores,
    'best_epoch': best_epoch_num,
    'best_rouge1': best_rouge1,
    'best_rouge2': best_rouge2,
    'all_rouge1': [r['rouge1'] for r in rouge_scores],
    'all_rouge2': [r['rouge2'] for r in rouge_scores]
}

with open('models/lstm_results.pkl', 'wb') as f:
    pickle.dump(lstm_results, f)
print(" LSTM результаты сохранены: lstm_results.pkl")

if tf_rouge1 > 0:
    transformer_results = {
        'rouge1': tf_rouge1,
        'rouge2': tf_rouge2,
        'predictions': tf_preds[:10],
        'references': tf_refs[:10]
    }
    
    with open('models/transformer_results.pkl', 'wb') as f:
        pickle.dump(transformer_results, f)
    print(" Transformer результаты сохранены: transformer_results.pkl")

comparison = {
    'lstm_rouge1': best_rouge1,
    'lstm_rouge2': best_rouge2,
    'transformer_rouge1': tf_rouge1,
    'transformer_rouge2': tf_rouge2,
    'improvement_rouge1': tf_rouge1 - best_rouge1,
    'improvement_rouge2': tf_rouge2 - best_rouge2,
    'best_model': 'transformer' if tf_rouge1 > best_rouge1 else 'lstm'
}

with open('models/comparison.pkl', 'wb') as f:
    pickle.dump(comparison, f)
print(" Сравнение сохранено: comparison.pkl")

examples = {
    'lstm_examples': [
        "feels alone because he has no friends on twitter et miley cyrus and i have to go",
        "for lunch today? im not a",
        "sandwiches oh has noes... hmmm didnt be a good"
    ],
    'transformer_examples': tf_preds[:3] if tf_preds else [],
    'references': [
        "doesnt answer him. what should",
        "tired of",
        "oh want to"
    ]
}

with open('models/examples.pkl', 'wb') as f:
    pickle.dump(examples, f)
print(" Примеры сохранены: examples.pkl")

print("\n" + "="*60)
print(" ФИНАЛЬНОЕ СРАВНЕНИЕ")
print("="*60)
print(f"\n{'Модель':<20} {'ROUGE-1':<12} {'ROUGE-2':<12}")
print("-"*44)
print(f"{'LSTM':<20} {best_rouge1:.4f}{'':8} {best_rouge2:.4f}")
print(f"{'DistilGPT2':<20} {tf_rouge1:.4f}{'':8} {tf_rouge2:.4f}")

if tf_rouge1 > 0:
    print(f"\n Улучшение Transformer:")
    print(f"   ROUGE-1: +{tf_rouge1 - best_rouge1:.4f}")
    print(f"   ROUGE-2: +{tf_rouge2 - best_rouge2:.4f}")

try:
    !cp -r models/ /content/drive/MyDrive/text-autocomplete/
    print(f"\n Все файлы скопированы в Google Drive")
except:
    print(f"\n Не удалось скопировать в Drive (не Colab или нет монтирования)")

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ LSTM И TRANSFORMER СОХРАНЕНЫ!")
print("="*60)